# Current development: real-voice pilot and useful-input workflow

**Goal:** obtain a recoverable vocal idea for the next component, preserve it when usable, compare only supported preparation, and explain when a clearer sung/hummed take is needed.

This notebook combines a **small completed original-recording learning pilot** with a **provisional acoustic review of a real HumTrans recording**. It does not certify musical correctness, infer intended notes, or turn arbitrary spoken text into a known intended melody. The newly trained small model failed its declared endpoint gates and is not adopted.

Run **Restart Kernel and Run All Cells** using **BPDR local Python**. Cells verify saved training and run one diagnostic CPU forward pass plus one short real-audio validation/review. **No training runs automatically.** Runtime files are written under `BPDR_Workspace/results/usefulness_notebook`; frozen datasets, checkpoints and historical notebooks remain intact.

Use [05: audible input/output evidence](05_panel_demonstration.ipynb) for the controlled pitch and noise examples. See [03](03_transformer_training.ipynb) and [04](04_transformer_evaluation.ipynb) for the earlier Transformer record. The new pilot below is a **LocalContextMLP, not a Transformer**.

In [ ]:
import os, sys, json, hashlib
from pathlib import Path
start=Path(os.environ.get('BPDR_COMPONENT_ROOT',Path.cwd())).resolve()
COMPONENT=next((p for p in (start,*start.parents) if p.name=='bpdr_input' and (p/'pyproject.toml').is_file()),None)
if COMPONENT is None: raise RuntimeError('Open inside BPDR_Workspace with the BPDR local Python kernel.')
WORKSPACE=COMPONENT/'BPDR_Workspace'
sys.path.insert(0,str(WORKSPACE/'source'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import torch
from IPython.display import display, Audio, Markdown
from bpdr_input.audio.pipeline import checksum

def read(path): return json.loads(Path(path).read_text(encoding='utf-8'))
PILOT=WORKSPACE/'results/small_real_pilot/1c94027926b5442e8f3e192afe530569'
if not (PILOT/'evidence.json').is_file():
    raise RuntimeError('The selected completed pilot is missing. Restore its saved evidence; this notebook does not start a new fit.')
D=read(PILOT/'declaration.json'); E=read(PILOT/'evidence.json')
RECORDS=read(PILOT/'selected_records.json')
WORKING_CARD=read(WORKSPACE/'selected_model/model.json')
WORKING_CHECKPOINT=WORKSPACE/'selected_model'/WORKING_CARD['checkpoint']
WORKING_SHA=checksum(WORKING_CHECKPOINT)
assert WORKING_SHA==WORKING_CARD['sha256']
print('Python:',sys.executable)
print('Workspace:',WORKSPACE)
print('Selected completed pilot:',PILOT)
print('No optimizer updates will run.')

## 1. Verify original-source evidence and the declared budget

Original Vocadito/Jamendo recordings provide real vocal acoustics. Their local detuning and boundary variants are constructed interventions, with known compensating targets. **Original does not mean certified correct singing**; performed pitch and model-estimated metadata do not establish the user's intended perfect melody.

The pilot selection used source identity, sign and corruption metadata rather than model scores. It used 32 training variants across eight singer/artist groups and 12 development variants across six different groups. Several variants share a recording. Jamendo artist identity is only a singer proxy. Previously inspected development material is not an untouched final test.

All three arms were declared together: the same seed/sampling, 180 optimizer updates or 90 seconds per arm, CPU, and the endpoint gate fixed before fitting. Incomplete runs and failed gates are ineligible. No automatic retry or longer training is unlocked.

In [ ]:
assert checksum(PILOT/'declaration.json')==E['declaration_sha256']
assert checksum(PILOT/'selected_records.json')==D['selected_records_sha256']
assert checksum(PILOT/'input_hashes.json')==E['input_hashes_sha256']
FROZEN=read(PILOT/'input_hashes.json')
for path,digest in FROZEN.items(): assert checksum(path)==digest,path
for source,archived in [('small_real_pilot.py','executed_small_real_pilot.py'),
                        ('local_context_pilot.py','executed_local_context_pilot.py')]:
    original=next(path for path in D['frozen'] if Path(path).name==source)
    assert checksum(PILOT/archived)==D['frozen'][original]
for condition,r in E['results'].items(): assert checksum(r['checkpoint'])==r['checkpoint_sha256']
train_groups={(r['dataset'],r['singer_id']) for r in RECORDS['training']}
dev_groups={(r['dataset'],r['singer_id']) for r in RECORDS['development']}
assert not train_groups & dev_groups
assert not {r['waveform_sha256']['A'] for r in RECORDS['training']} & {r['waveform_sha256']['A'] for r in RECORDS['development']}
rows=[]
for split,records in RECORDS.items():
    for ds in ['vocadito','jamendo']:
        part=[r for r in records if r['dataset']==ds]
        rows.append(dict(split=split,dataset=ds,dependent_variants=len(part),
            original_recordings=len({r['phrase_id'] for r in part}),
            singer_or_artist_groups=len({r['singer_id'] for r in part})))
display(pd.DataFrame(rows))
print('Verified',len(FROZEN),'frozen input/source files and every pilot checkpoint.')
print('No training/development source-group or original-audio checksum overlap.')
print('Declared model:',D['model'])
print('Declared stop:',D['wall_clock_rule'])
print('Declared gate:',D['gate'])
print('Natural intended-note truth:',D['natural_error_truth'],'Research-ready examples:',D['research_ready_examples'])

## 2. What the small model learned from

The `LocalContextMLP` uses 31 fixed-scale features derived from one recording's own pitch, confidence, explicit voicing, energy/onset and local context. The BPDR/augmentation arms also receive a boundary feature; the frame-only arm omits it. Context includes circular chromatic residuals and fixed rolling windows. This is not a named-key estimator or a correct-note oracle.

There is no clean paired reference, intervention support, correction target, group identity or annotation melody in the inference interface. Known targets and masks enter losses/scoring only. A/B anchor original preservation, C/D teach response to known detuning, and BPDR adds boundary consistency plus counter-detuning response.

The small model is an additional bounded diagnostic. It does not replace the proposal's Transformer experiment or establish a research advantage.

In [ ]:
display(pd.DataFrame([{'condition':condition,'model':r['model']['type'],
    'parameters':r['model']['parameters'],'features':r['model']['features'],
    'completed_updates':r['completed_updates'],'training_seconds':r['training_seconds'],
    'completed_budget':r['completed_budget'],'eligible_for_demo':r['eligible_for_demo']}
    for condition,r in E['results'].items()]))
fig,ax=plt.subplots(figsize=(10,4))
for condition in D['conditions']:
    history=read(PILOT/(condition+'_history.json'))
    ax.plot([r['update'] for r in history],[r['loss'] for r in history],label=condition)
ax.set(xlabel='Optimizer update',ylabel='Recorded weighted training objective',title='Saved bounded learning curves: lower loss is not validation')
ax.legend();plt.tight_layout();plt.show()
print('Forbidden inference inputs:',D['forbidden_inputs'])
print('Actual recorded optimizer:',D['optimizer'])

## 3. Recompute saved results and display every arm

Correction MAE compares predictions with the known compensating shift; zero MAE is the error from doing nothing. Response near one is desirable and response near zero can indicate ignoring detuning. Original alteration measures unwanted changes to A/B. Metrics first average dependent variants within source groups, then give groups equal weight; response is the median of group medians.

The engineering endpoint requires **each dataset** to have response ≥0.25, original alteration ≤1 cent and positive correction gain. These are declared pilot checks, not musical approval. We recompute the summaries from every saved per-variant row rather than transcribing favorable numbers.

In [ ]:
from bpdr_input.experiments.learning.small_real_pilot import summarise
RESULT_ROWS=[]
for condition,result in E['results'].items():
    for split in ['training','development']:
        rows=read(PILOT/(condition+'_'+split+'_rows.json'))
        recomputed=summarise(rows)
        for dataset,metrics in result[split]['by_dataset'].items():
            for key,value in metrics.items():
                expected=recomputed['by_dataset'][dataset][key]
                if isinstance(value,bool): assert value==expected
                else: np.testing.assert_allclose(value,expected,rtol=1e-6,atol=1e-7)
            RESULT_ROWS.append(dict(condition=condition,split=split,dataset=dataset,**metrics))
TABLE=pd.DataFrame(RESULT_ROWS)
display(TABLE[['condition','split','dataset','records','independent_groups','original_recordings',
    'correction_mae','zero_mae','median_response_gain','unshifted_alteration','correction_gain','passes']])
fig,axes=plt.subplots(2,2,figsize=(13,8))
for row,dataset in enumerate(['vocadito','jamendo']):
    part=TABLE[(TABLE.split=='development')&(TABLE.dataset==dataset)]
    axes[row,0].bar(part.condition,part.median_response_gain,color='#477b88')
    axes[row,0].axhline(.25,color='grey',ls=':');axes[row,0].set(title=dataset+' development response',ylabel='Gain (ideal 1)')
    axes[row,1].bar(part.condition,part.unshifted_alteration,color='#a3604a')
    axes[row,1].axhline(1,color='grey',ls=':');axes[row,1].set(title=dataset+' original preservation',ylabel='Original alteration (cents)')
for ax in axes.ravel(): ax.tick_params(axis='x',rotation=12)
plt.tight_layout();plt.show()
print('Eligible conditions:',E['eligible_conditions'])
print('Adopted:',E['model_adopted'],'Final test accessed:',E['final_test_audio_accessed'])
assert not E['eligible_conditions'] and not E['model_adopted']
print('Outcome: stop this declared pilot; no larger fit and no model replacement.')

The BPDR arm responded to some Jamendo interventions, but its original alteration exceeded the fixed preservation limit. Vocadito response did not generalize adequately. **All three arms failed eligibility.** This is evidence about this small setup, not proof that the recordings are musically wrong or that BPDR is impossible.

## 4. Diagnostic forward pass and refusal to adopt

We load the checksum-verified small BPDR checkpoint for one diagnostic computation and compare it with saved predictions. We then exercise the separate deployment loader, which must refuse this failed candidate. A valid neural forward pass is not sufficient for adoption.

In [ ]:
from bpdr_input.modeling.local_context_pilot import load_checkpoint, load_candidate, predict_raw
torch.set_num_threads(2)
record=RECORDS['development'][0]
with np.load(Path(record['input_directory'])/'C_input.npz') as archive:
    raw=archive['boundary_features_71'].copy()
small_model,payload=load_checkpoint(E['results']['bpdr']['checkpoint'])
new_prediction=predict_raw(small_model,raw)
with np.load(PILOT/'bpdr_development_predictions.npz') as archive:
    saved=archive['0_C'].copy()
np.testing.assert_allclose(new_prediction,saved,rtol=1e-5,atol=1e-5)
with np.load(record['target_path']) as archive:
    target=archive['correction_target_cents'].copy()
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(target,label='Known compensating target',color='#a3604a')
ax.plot(new_prediction,label='Small BPDR diagnostic prediction',color='#477b88')
ax.set(xlabel='Aligned frame index',ylabel='Correction (cents)',title='One real-source development intervention: '+record['phrase_id'])
ax.legend();plt.tight_layout();plt.show()
print('Diagnostic inference finite:',np.isfinite(new_prediction).all(),'Shape:',new_prediction.shape)
try:
    load_candidate(E['results']['bpdr']['checkpoint'])
except ValueError as exc:
    print('Expected deployment refusal:',exc)
else:
    raise AssertionError('An ineligible model was accepted for deployment')
assert checksum(WORKING_CHECKPOINT)==WORKING_SHA
print('Historical working Transformer unchanged; failed MLP is diagnostic only.')

## 5. Real humming: technical validation followed by provisional evidence review

The next cells use the original **HumTrans F01 recording**, without deliberately adding detuning or noise. Its dataset provenance says humming; that is the fixture's declared intention, not a participant's answer or a learned classification. Reference MIDI is not supplied to runtime analysis.

The pipeline preserves the source bytes and creates an untrimmed mono 22,050 Hz working copy. Because the original is 44,100 Hz, sample counts change during resampling; duration, pitch/timing policy and source preservation are checked. A technical acceptance remains separate from acoustic usefulness.

The provisional acoustic policy asks for agreeing confidence/voicing, some total and contiguous two-tracker support, and majority agreement among primary voiced frames. It deliberately has no minimum pitch variation, note count or whole-recording voiced fraction. Repeated notes, chant, slides and pauses are allowed. The settings are fixed demo engineering values, **not validated classification thresholds**.

In [ ]:
from bpdr_input.repair.demo_runtime import run as run_demo
from bpdr_input.repair.demo_review import review, export_reviewed
from bpdr_input.audio.usability import assess, UsabilityPolicy
intake=read(COMPONENT/'data/humtrans/intake_receipt.json')
entry=next(r for r in intake['records'] if r['key']=='F01_0001_0001_1_D')
SOURCE=Path(entry['path'])
assert checksum(SOURCE)==entry['sha256']
RUNTIME,REPORT=run_demo(SOURCE,method='Validate and preserve',output_root=WORKSPACE/'results/usefulness_notebook')
assert REPORT['handoff']['status']=='ACCEPT_UNCHANGED'
assert not REPORT['prepared_handoff_enabled']
assert checksum(RUNTIME/'original.wav')==entry['sha256']
ASSESSMENT=review(RUNTIME,REPORT,intent='singing_humming_chant')
assert not ASSESSMENT['shared_handoff_changed']
accepted=REPORT['handoff']['accepted_audio']
assert checksum(accepted['path'])==accepted['checksum']
assert sf.info(SOURCE).duration==accepted['duration_seconds']
display(pd.DataFrame([{'technical_status':REPORT['handoff']['status'],
    'provisional_evidence':ASSESSMENT['original']['decision'],
    'musical_correctness_assessed':ASSESSMENT['original']['musical_correctness_assessed'],
    'handoff_approval':ASSESSMENT['original']['handoff_approval'],
    'source_rate_hz':sf.info(SOURCE).samplerate,'working_rate_hz':accepted['sample_rate_hz'],
    'duration_seconds':accepted['duration_seconds']}]))
print('Provisional feedback:',ASSESSMENT['original']['feedback'])
display(pd.DataFrame([ASSESSMENT['original']['observations']]))
display(pd.DataFrame([ASSESSMENT['original']['policy']]))
for label,path in [('Original HumTrans humming',SOURCE),('Technically accepted working copy: preserved vocal idea',accepted['path'])]:
    display(Markdown('**'+label+'**'));display(Audio(filename=str(path),normalize=False))
print('New local runtime evidence:',RUNTIME)

## 6. Intention must be declared; pitch alone cannot determine it

The following is an **interface branch test on the same known humming measurements**, not a speech dataset or classifier evaluation. We vary only the metadata declaration to expose the different decisions. Declared ordinary reading means no intended tune was supplied. Chanted or speech-like singing with a musical idea belongs in the singing/humming/chant option.

Unknown intent should request confirmation, rather than deciding that a pitched voice must be singing. Ordinary spoken text without a tune should request a hummed/sung rendition. Silence, missing acoustic evidence and technical failure have separate reasons. None should be described as a judgment of singing skill.

In [ ]:
with np.load(RUNTIME/'features.npz') as archive:
    features={key:archive[key] for key in archive.files}
from bpdr_input.repair.rule_correction import fine_hz
wave,rate=sf.read(accepted['path'],dtype='float32')
secondary=fine_hz(wave,rate,features['times_seconds'])
intent_rows=[]
for declared in ['singing_humming_chant','reading_spoken_text','unsure']:
    result=assess(features,REPORT['handoff']['status'],declared,independent_f0_hz=secondary,duration_seconds=len(wave)/rate)
    intent_rows.append({'metadata_scenario':declared,'provisional_decision':result['decision'],
        'next_step':result['next_step'],'feedback':result['feedback']})
display(pd.DataFrame(intent_rows))
assert intent_rows[1]['provisional_decision']=='NEED_CLEARER_INPUT'
assert intent_rows[2]['provisional_decision']=='NEED_INTENT_CONFIRMATION'
try:
    export_reviewed(RUNTIME,REPORT,ASSESSMENT,confirmed=False)
except ValueError as exc:
    print('Expected unconfirmed-export refusal:',exc)
else:
    raise AssertionError('Export proceeded without explicit intention confirmation')
assert not (RUNTIME/'reviewed_handoffs').exists()
print('No participant decision was simulated and no reviewed export was created by this notebook.')

## 7. Verified connection to the next component

This section **reads an already completed automated integration fixture**. It does not run GAME again. The fixture has `user_confirmed=False` and `confirmation_kind=automated_test_fixture`; it does not represent a participant's musical approval.

The reviewed package contains the unchanged real HumTrans working waveform. The pinned GAME extractor consumed that accepted file and produced CSV/MIDI in a fresh attempt directory. We verify the accepted waveform, model and output checksums before displaying the result. This demonstrates a working connection and traceable artifacts, **not intended-melody accuracy or an improvement from correction**.

In [ ]:
FIXTURE_PATH=WORKSPACE/'results/demo_sessions/integration_fixture_current.json'
FIXTURE=read(FIXTURE_PATH)
REPO=next(p for p in WORKSPACE.parents if (p/'J26-DS-339').is_dir())
def fixture_path(path):
    candidate=Path(path)
    return (candidate if candidate.is_absolute() else REPO/candidate).resolve()
exported=fixture_path(FIXTURE['export_directory'])
assert exported.is_relative_to(WORKSPACE.resolve())
diagnostics=read(exported/'diagnostics.json')
assert diagnostics['confirmation_kind']=='automated_test_fixture'
assert not diagnostics['user_confirmed'] and not diagnostics['research_approval']
fixture_audio=fixture_path(FIXTURE['handoff']['accepted_audio']['path'])
assert checksum(fixture_audio)==FIXTURE['handoff']['accepted_audio']['checksum']==FIXTURE['extractor']['accepted_sha256']
assert FIXTURE['extractor']['success']
GAME_MODEL=COMPONENT/'checkpoints/melody-extraction/models/GAME-1.0-medium/model.pt'
assert checksum(GAME_MODEL)==FIXTURE['extractor']['model_sha256']
for artifact in FIXTURE['extractor']['artifacts']:
    assert checksum(artifact['path'])==artifact['sha256']
csv=next(Path(a['path']) for a in FIXTURE['extractor']['artifacts'] if Path(a['path']).suffix=='.csv')
notes=pd.read_csv(csv)
display(pd.DataFrame([{'scope':FIXTURE['scope'],'source':'Original real HumTrans humming',
    'accepted_status':FIXTURE['handoff']['status'],'extractor_succeeded':FIXTURE['extractor']['success'],
    'extractor_elapsed_seconds':FIXTURE['extractor']['elapsed_seconds'],
    'extracted_events':len(notes),'participant_confirmation':diagnostics['user_confirmed'],
    'intended_melody_accuracy_measured':False}]))
display(notes.head(12))
fig,ax=plt.subplots(figsize=(11,3.5))
for _,note in notes.iterrows(): ax.hlines(note['pitch'],note['onset'],note['offset'],linewidth=3,color='#477b88')
ax.set(xlabel='Seconds on accepted waveform',ylabel='GAME numeric pitch',
    title='Actual downstream extraction: connection evidence, not intended-note scoring')
plt.tight_layout();plt.show()
print('Verified completed fixture:',FIXTURE_PATH)
print('Verified CSV:',csv)
print('No new extractor execution and no real-user musical approval claimed.')

## 8. Operational meaning and remaining validation

| Stage | Current behavior | Honest limit |
| --- | --- | --- |
| Technical intake | Preserve bytes; inspect format, duration, overload and silence; create working copy | Accepted file does not prove usable music |
| Provisional review | Record declared intent and inspect two-tracker support | Not a calibrated singing/speech or intention classifier |
| Optional preparation | Existing preview methods retain the original and audit candidates | A numerical audit does not certify artistic correctness |
| User selection | Require explicit confirmation before demo-only reviewed export | A scripted fixture is not real participant evidence |
| Downstream check | A verified reviewed package can be passed to the existing extractor | Extractability does not establish correct intended melody |
| Small original-data learning | Completed bounded pilot with all arms shown | Failed all declared eligibility gates; no new model adopted |

A user may choose to proceed with an unchanged useful vocal. If uncertainty or interference prevents recoverable melodic evidence, feedback should ask for a more usable take and explain why. The system should not force unfamiliar notes into a scale or manufacture the missing intention of ordinary reading.

Next research work needs reviewed useful/unusable examples, intended-melody information where available, unfamiliar singer/melody evaluation, baseline comparisons, downstream utility measures and user/listener preservation judgments. **Research-ready examples remain zero.** The demonstrated workflow is an engineering prototype with explicit uncertainty.

In [ ]:
assert checksum(WORKING_CHECKPOINT)==WORKING_SHA
for condition,result in E['results'].items():
    assert checksum(result['checkpoint'])==result['checkpoint_sha256']
assert checksum(SOURCE)==entry['sha256']
SUMMARY={'pilot':str(PILOT),'pilot_evidence_sha256':checksum(PILOT/'evidence.json'),
    'all_pilot_conditions_ineligible':not bool(E['eligible_conditions']),
    'working_checkpoint_unchanged':checksum(WORKING_CHECKPOINT)==WORKING_SHA,
    'original_source_unchanged':checksum(SOURCE)==entry['sha256'],
    'runtime_directory':str(RUNTIME),'technical_status':REPORT['handoff']['status'],
    'provisional_evidence':ASSESSMENT['original']['decision'],
    'user_reviewed_export_created_by_notebook':False,'new_optimizer_updates_in_notebook':0,
    'natural_intention_accuracy_claimed':False}
print(json.dumps(SUMMARY,indent=2))
print('Completed current-development walkthrough; failed training remains visible and no model was adopted.')